# CobraBox Use-Case :: Analysis and statistics _ interictal iEEG Zurich dataset

Authors: *[COBRA group](https://cobra.cs.cas.cz), Institute of Computer Science, The Czech Academy of Sciences*

<div align="left">
<img src="Images/Logo_CAS_ICS.png" align="left" width="254" alt="logo ICS">
</div>


<br>
<br>

---------------------

This is the third of the three notebooks that together form a complete, reproducible example of using [CobraBox](https://github.com/cobragroup/cobrabox) on an intracranial EEG (iEEG) dataset:

1. ***Getting Started & Data Exploration*** introduces the toolbox and walks, step-by-step, through the example dataset ([interictal iEEG Zurich dataset](https://openneuro.org/datasets/ds003498/versions/1.1.1)). It is read-only.
2. ***Data preprocessing and preparation*** turns the raw recordings of the [interictal iEEG Zurich dataset](https://openneuro.org/datasets/ds003498/versions/1.1.1) into clean, segmented, analysis-ready data saved to disk.
3. ***Analysis and statistics (this notebook)*** computes directed connectivity, derives inward and outward strength, runs the statistics, and produces the main result.

Notebooks 2 and 3 follow the pipeline of the accompanying study (Stergiadis C, Halliday DM, Kazis D, Klados MA. *High-frequency directed networks can identify epileptogenic tissue and predict surgical outcome in drug-resistant epilepsy*. Epilepsy Research. 2026;226:107838. doi: [10.1016/j.eplepsyres.2026.107838](https://doi.org/10.1016/j.eplepsyres.2026.107838)).

<span style="color:darkred">
IMPORTANT: As in notebook 2, the routine steps that are <b>not</b> the point of this use-case (loading the patient metadata, loading the saved segments, and reading/writing intermediate files) are handled by the local helper module <code>local_utils.py</code> and called explicitly as <code>local_utils.&lt;function&gt;()</code>. The analysis itself (inward/outward strength, the statistics and the figure) is written out in full in the notebook. Note the relative import <code>import local_utils</code> (not <code>from local_utils import *</code>).
</span>

***IMPORTANT***: For this notebook to run, you first ***must*** run notebook 2 (02_preprocessing_and_computation.ipynb)

### Contents of this notebook

1. Directed connectivity (DTF)
2. Inward and outward strength
3. Inside vs outside the resection
4. Statistical testing
5. Visualising the result
6. Where we end up …

### Import dependencies

Running this notebook requires ***python*** (>=3.11), ***xarray*** (>2026.2.0) and ***cobrabox***
(>=X.Y). The analysis and figure additionally use ***NumPy***, ***pandas***, ***SciPy*** and
***Matplotlib***. The routine helpers live in the local module ***local_utils.py*** that ships
alongside these notebooks.

In [ ]:
## IMPORT THE PACKAGES NEEDED TO RUN THE NOTEBOOK
# Python standard library imports
from pathlib import Path

# Third-party imports
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

%matplotlib inline
from scipy.stats import wilcoxon

import cobrabox as cb

# Local libraries and modules
import local_utils

### What this notebook computes

Using the segments saved by notebook 2 (02_preprocessing_and_computation.ipynb), we reproduce part of the results of (Stergiadis et al., 2026), up to the point
that concerns us here: whether **resected (epileptogenic) tissue behaves differently from non-resected (non-epileptogenic)** in terms of *directed* connectivity.

Concretely, for every contact we measure how strongly it **sends** influence to the rest of the
network (**outward strength**) and how strongly it **receives** influence (**inward strength**). We
then compare these values **inside** versus **outside** the resection, separately for patients with a
**good** and a **poor** surgical outcome.

> **What the study found (Stergiadis et al., 2026, §3.2).** In good-outcome patients, resected tissue
> showed **lower outward strength** than the rest of the brain in HFO-free segments, most clearly at the higher frequency
> bands; inward strength showed a similar but non-significant trend, and poor-outcome patients showed no such
> difference. This fits the idea that epileptogenic tissue is *functionally isolated* by the surrounding network
> during the quiet period between seizures (interictal period).

The pipeline here is: directed connectivity → inward/outward strength → per-patient medians inside/outside
→ a paired statistical test → a figure.

## 3. Inside vs outside the resection

We now summarise each patient with two values per band: the **median** strength of
the contacts **inside** the resection and the **median** of those **outside**. The median (rather than
the mean) is used because some patients have only a few electrodes, where a single outlier could
easily distort the average.

In [ ]:
# Long -> per-patient medians, one row per (subject, band, measure) with inside/outside columns
long = strength_df.melt(
    id_vars=["subject", "outcome", "band", "region"],
    value_vars=["in_strength", "out_strength"],
    var_name="measure",
    value_name="value",
)

medians = (
    long.groupby(["subject", "outcome", "band", "measure", "region"])["value"]
    .median()
    .reset_index()
    .pivot_table(index=["subject", "outcome", "band", "measure"], columns="region", values="value")
    .reset_index()
)
medians.columns.name = None
for col in ("inside", "outside"):
    if col not in medians.columns:
        medians[col] = np.nan

medians.head(8)

## 4. Statistical testing

Within each outcome group (good / poor), for each band and each measure, we compare the per-patient
**inside** and **outside** medians with a **paired Wilcoxon signed-rank test** (a non-parametric test
suited to small, paired samples). Because we test eight bands at once, we correct the p-values with the
**Benjamini–Hochberg** false-discovery-rate procedure. A result is called significant when the
corrected value `q < 0.05`.

In [ ]:
def bh_fdr(pvals):
    """Benjamini-Hochberg FDR correction; NaNs are preserved."""
    p = np.asarray(pvals, dtype=float)
    mask = ~np.isnan(p)
    q = np.full_like(p, np.nan)
    pv = p[mask]
    m = pv.size
    if m:
        order = np.argsort(pv)
        adj = pv[order] * m / (np.arange(m) + 1)
        adj = np.minimum.accumulate(adj[::-1])[::-1]
        out = np.empty(m)
        out[order] = np.clip(adj, 0, 1)
        q[mask] = out
    return q


band_names = list(BANDS)
results = []
for outcome in ("good", "poor"):
    for measure in ("out_strength", "in_strength"):
        recs = []
        sub = medians[(medians["outcome"] == outcome) & (medians["measure"] == measure)]
        for band in band_names:
            b = sub[sub["band"] == band].dropna(subset=["inside", "outside"])
            if len(b) < 2:
                p = np.nan
            else:
                try:
                    _, p = wilcoxon(b["inside"].values, b["outside"].values)
                except ValueError:
                    p = np.nan
            recs.append(
                {
                    "outcome": outcome,
                    "measure": measure,
                    "band": band,
                    "n_patients": len(b),
                    "p_value": p,
                }
            )
        for r, q in zip(recs, bh_fdr([r["p_value"] for r in recs])):
            r["q_value"] = q
            r["significant"] = bool(q < 0.05) if not np.isnan(q) else False
            results.append(r)

stats_df = pd.DataFrame(results)
pd.set_option("display.float_format", "{:.4f}".format)
print(stats_df.to_string(index=False))

## 5. Visualising the result

For each measure we plot, per band, the per-patient **inside** (orange) and **outside** (blue)
medians, with a **Good** and a **Poor** block. Grey lines link the two values of the same patient, and
a `*` marks bands that survive the FDR correction.

In [ ]:
def strength_plot(measure, measure_label):
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    for ax, band in zip(axes.flatten(), band_names):
        for grp, x0 in (("good", 0.0), ("poor", 2.6)):
            b = medians[
                (medians["outcome"] == grp)
                & (medians["measure"] == measure)
                & (medians["band"] == band)
            ].dropna(subset=["inside", "outside"])
            inside, outside = b["inside"].values, b["outside"].values
            xin, xout = x0, x0 + 1.0
            for a, c in zip(inside, outside):
                ax.plot([xin, xout], [a, c], color="gray", alpha=0.4, lw=0.7)
            ax.scatter(np.full(len(inside), xin), inside, color="tomato", s=18, zorder=3)
            ax.scatter(np.full(len(outside), xout), outside, color="steelblue", s=18, zorder=3)
            if len(inside):
                ax.plot([xin - 0.18, xin + 0.18], [np.median(inside)] * 2, color="black", lw=2)
            if len(outside):
                ax.plot([xout - 0.18, xout + 0.18], [np.median(outside)] * 2, color="black", lw=2)
            row = stats_df[
                (stats_df["outcome"] == grp)
                & (stats_df["measure"] == measure)
                & (stats_df["band"] == band)
            ]
            if len(row) and bool(row["significant"].values[0]):
                ax.text((xin + xout) / 2, 1.04, "*", ha="center", va="bottom", fontsize=15)
        ax.set_title(band, fontsize=9)
        ax.set_xticks([0.5, 3.1])
        ax.set_xticklabels(["Good", "Poor"], fontsize=9)
        ax.set_ylim(-0.05, 1.15)
        ax.set_ylabel(measure_label, fontsize=8)

    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="tomato",
            markersize=8,
            label="inside resection",
        ),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="steelblue",
            markersize=8,
            label="outside resection",
        ),
    ]
    fig.legend(
        handles=handles, loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.02)
    )
    fig.suptitle(
        f"{CONNECTIVITY_METHOD.upper()} {measure_label}: inside vs outside resection",
        fontsize=13,
        y=1.01,
    )
    plt.tight_layout()
    plt.show()


strength_plot("out_strength", "outward strength")  # headline result
strength_plot("in_strength", "inward strength")  # shown for comparison

## 6. Where we end up

Reading the table and the figure together tells us, band by band, whether resected tissue differs from
the rest of the brain in how it drives (outward) or receives (inward) directed influence, and whether
that pattern is specific to good-outcome patients.

If the pipeline reproduces (Stergiadis et al., 2026), the clearest signal is **lower outward strength inside the
resection of good-outcome patients** at the higher bands, with no comparable effect in poor-outcome
patients. Two things to keep in mind when comparing with the paper: this notebook uses **DTF** (the
study used the closely related **dDTF**), and it draws its segments **at random** rather than
separating HFO-free data (see notebook 2). The aim here is to show how such an analysis is built with
CobraBox, not to reproduce the paper exactly.